# TenderMind — موديلات Ollama على كارت Colab

البرنامج بيشتغل على جهازك، والموديلات (`qwen2.5:3b` و`qwen3:4b`) بتشتغل هنا على كارت T4،
ورا بوابة بباسورد: أي طلب من غير الباسورد بيترفض، ومسموح بس بالطلبات اللي البرنامج بيستخدمها.

**الخطوات**
1. اتأكد إن الـruntime عليه GPU: **Runtime ← Change runtime type ← T4 GPU ← Save**.
2. **Runtime ← Run all** (أول مرة بتاخد كام دقيقة لتحميل الموديلات، حوالي 4.5 جيجا).
3. الخلية 8 هتطبع سطر بيبدأ بـ`OLLAMA_BASE_URL=https://tm:...`.
   **انسخه والزقه في ملف `D:\TenderMind\.env` على جهازك** (الأمر: `notepad D:\TenderMind\.env`) واحفظ.
   🔴 **متبعتش السطر ده في الشات** — فيه الباسورد.
4. **سيب التاب ده مفتوح** طول ما إنت بتجرّب. لو الجلسة اتقفلت، الرابط بيموت.

**تنبيهات**
- كل ما تشغّل الخلية 7 من تاني بيطلع **رابط جديد**، وكل ما تشغّل الخلية 6 من تاني بيطلع **باسورد جديد**؛
  في الحالتين شغّل الخلية 8 وحدّث السطر في `.env`.
- الجلسة المجانية بتقفل بعد ١٢ ساعة بالكتير.
- لما تخلص: الخلية 10 بتقفل الرابط، أو **Runtime ← Disconnect and delete runtime**.


In [ ]:
# 1) GPU check + helpers
import json, os, re, secrets, shutil, subprocess, sys, time
import requests

LOCAL = "http://127.0.0.1:11434"
OLLAMA_VERSION = "0.12.3"                 # same image tag as docker-compose.yml (production)
MODELS = ["qwen2.5:3b", "qwen3:4b"]       # same pulls as docker-compose.yml (ollama-models)


def sh(cmd, timeout=1800):
    """Run a shell command; stop the notebook loudly if it fails."""
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout)
    out = (r.stdout or "") + (r.stderr or "")
    if r.returncode != 0:
        print(out[-4000:])
        raise RuntimeError(f"command failed (exit {r.returncode}): {cmd}")
    return out


def tail(path, n=3000):
    return open(path).read()[-n:] if os.path.exists(path) else f"({path} not found)"


gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError("No GPU on this runtime. Runtime > Change runtime type > T4 GPU > Save, then Run all again.")
print("GPU:", gpu.stdout.strip())


# The gate script written to /content/ollama_gate.py by cell 6
GATE_CODE = r'''"""Password gate in front of Ollama: Basic auth on every request, only the endpoints TenderMind uses."""
import base64
import hmac
import http.client
import os
import sys
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

USER = os.environ["GATE_USER"]
PASSWORD = os.environ["GATE_PASSWORD"]
PORT = int(os.environ.get("GATE_PORT", "11435"))
UPSTREAM_PORT = int(os.environ.get("GATE_UPSTREAM_PORT", "11434"))
EXPECTED = ("Basic " + base64.b64encode(f"{USER}:{PASSWORD}".encode()).decode()).encode()
ALLOWED = {("GET", "/api/tags"), ("GET", "/api/version"), ("POST", "/api/chat"), ("POST", "/api/generate")}
SKIP_HEADERS = {"connection", "keep-alive", "transfer-encoding", "te", "trailer", "upgrade",
                "proxy-authenticate", "proxy-authorization", "content-length", "date", "server"}


class Gate(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, fmt, *args):
        # request line and status only; the Authorization header is never logged
        sys.stdout.write("%s %s\n" % (self.log_date_time_string(), fmt % args))
        sys.stdout.flush()

    def _reply(self, code, body=b"", headers=()):
        self.send_response(code)
        for k, v in headers:
            self.send_header(k, v)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _handle(self):
        # Read the body before anything else, even for a request that will be refused: cloudflared
        # reuses connections, and an unread body would be parsed as the start of the next request.
        if "chunked" in self.headers.get("Transfer-Encoding", "").lower():
            self.close_connection = True
            return self._reply(411, b"send a Content-Length\n", [("Connection", "close")])
        length = int(self.headers.get("Content-Length") or 0)
        body = self.rfile.read(length) if length else None
        given = self.headers.get("Authorization", "").encode("utf-8", "replace")
        if not hmac.compare_digest(given, EXPECTED):
            return self._reply(401, b"unauthorized\n", [("WWW-Authenticate", 'Basic realm="ollama"')])
        if (self.command, self.path.split("?", 1)[0]) not in ALLOWED:
            return self._reply(403, b"endpoint not allowed\n")
        conn = http.client.HTTPConnection("127.0.0.1", UPSTREAM_PORT, timeout=600)
        try:
            conn.request(self.command, self.path, body=body, headers={
                "Host": f"localhost:{UPSTREAM_PORT}",
                "Content-Type": self.headers.get("Content-Type", "application/json")})
            resp = conn.getresponse()
            data = resp.read()
            keep = [(k, v) for k, v in resp.getheaders() if k.lower() not in SKIP_HEADERS]
            self._reply(resp.status, data, keep)
        except Exception as e:
            self._reply(502, f"upstream error: {type(e).__name__}\n".encode())
        finally:
            conn.close()

    # every method goes through the same checks (HEAD keeps the stdlib 501, which closes the connection)
    do_GET = do_POST = do_PUT = do_PATCH = do_DELETE = do_OPTIONS = _handle


if __name__ == "__main__":
    ThreadingHTTPServer(("127.0.0.1", PORT), Gate).serve_forever()
'''

In [ ]:
# 2) Install Ollama, pinned to the production version
if shutil.which("ollama") and OLLAMA_VERSION in sh("ollama --version"):
    print(f"Ollama {OLLAMA_VERSION} already installed")
else:
    sh("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null")
    sh("curl -fsSL -o /content/ollama-install.sh https://ollama.com/install.sh")   # a failed download stops here
    sh(f"OLLAMA_VERSION={OLLAMA_VERSION} sh /content/ollama-install.sh")
    ver = sh("ollama --version")
    print(ver.strip())
    if OLLAMA_VERSION not in ver:
        raise RuntimeError(f"expected Ollama {OLLAMA_VERSION}, got: {ver.strip()}")

In [ ]:
# 3) Start the Ollama server on the GPU (reused if it is already running with the right version)
OLLAMA_LOG = "/content/ollama.log"


def ollama_version():
    try:
        r = requests.get(LOCAL + "/api/version", timeout=3)
        return r.json().get("version") if r.status_code == 200 else None
    except (requests.RequestException, ValueError):
        return None


running = ollama_version()
if running and running != OLLAMA_VERSION:
    raise RuntimeError(f"an Ollama server {running} is already running (need {OLLAMA_VERSION}). "
                       "Runtime > Disconnect and delete runtime, then Run all again.")
if running:
    print(f"Ollama {running} is already running")
else:
    env = dict(os.environ,
               OLLAMA_HOST="127.0.0.1:11434",
               OLLAMA_KEEP_ALIVE="-1",      # keep both models loaded between the app's requests
               LD_LIBRARY_PATH="/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", ""))
    # start_new_session: stopping a cell sends SIGINT to the kernel's whole process group, which
    # killed the tunnel in the first live run; the background services get their own session.
    ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=open(OLLAMA_LOG, "w"),
                                   stderr=subprocess.STDOUT, env=env, start_new_session=True)
    for _ in range(90):
        if ollama_version():
            break
        if ollama_proc.poll() is not None:
            raise RuntimeError("ollama serve exited:\n" + tail(OLLAMA_LOG))
        time.sleep(1)
    else:
        raise RuntimeError("Ollama did not answer within 90 s:\n" + tail(OLLAMA_LOG))
    print(f"Ollama {ollama_version()} is up")

compute = [l for l in tail(OLLAMA_LOG, 200000).splitlines() if "inference compute" in l]
print("\n".join(compute) or "(no 'inference compute' line in the log)")

In [ ]:
# 4) Download the two models the app uses (about 4.5 GB, a few minutes)
for m in MODELS:
    sh(f"ollama pull {m}")
have = [x["name"] for x in requests.get(LOCAL + "/api/tags", timeout=10).json().get("models", [])]
missing = [m for m in MODELS if m not in have]
if missing:
    raise RuntimeError(f"models missing after pull: {missing} (have: {have})")
print("models ready:", have)

In [ ]:
# 5) Load both models on the GPU with the same request shape the app sends, and check the answers
PROBE = 'Reply with only this JSON object: {"ok": true}'


def app_chat(base, model, auth=None, timeout=300):
    """Same payload as app/engines/tender_bridge.py and app/matchers/ollama_matcher.py."""
    payload = {"model": model, "stream": False, "format": "json", "options": {"temperature": 0},
               "messages": [{"role": "user", "content": PROBE}]}
    if model.startswith("qwen3"):
        payload["think"] = False
    t0 = time.time()
    r = requests.post(base + "/api/chat", json=payload, auth=auth, timeout=timeout)
    r.raise_for_status()
    content = ((r.json().get("message") or {}).get("content") or "").strip()
    try:
        parsed = json.loads(content)
    except ValueError:
        raise RuntimeError(f"{model} did not return JSON: {content[:200]!r}")
    if not isinstance(parsed, dict) or parsed.get("ok") not in (True, "true"):
        raise RuntimeError(f"{model} returned unexpected JSON: {content[:200]!r}")
    return time.time() - t0, content


for m in MODELS:
    first, _ = app_chat(LOCAL, m)
    warm, reply = app_chat(LOCAL, m)
    print(f"{m}: first call {first:.1f}s (includes loading), warm call {warm:.1f}s, reply {reply[:60]}")

ps = sh("ollama ps")
print(ps)
on_gpu = [l for l in ps.splitlines() if "100% GPU" in l]
if len(on_gpu) < len(MODELS):
    raise RuntimeError("Not every model is fully on the GPU (see the PROCESSOR column). Check /content/ollama.log.")

In [ ]:
# 6) Password gate in front of Ollama (a rerun replaces the gate AND the password)
GATE_SCRIPT, GATE_LOG, GATE_PORT = "/content/ollama_gate.py", "/content/gate.log", 11435
GATE = f"http://127.0.0.1:{GATE_PORT}"
open(GATE_SCRIPT, "w").write(GATE_CODE)

subprocess.run(["pkill", "-f", GATE_SCRIPT])          # no shell, so pkill cannot match its own parent
time.sleep(1)
GATE_USER, GATE_PASSWORD = "tm", secrets.token_urlsafe(24)
gate_proc = subprocess.Popen([sys.executable, GATE_SCRIPT], stdout=open(GATE_LOG, "w"), stderr=subprocess.STDOUT,
                             env=dict(os.environ, GATE_USER=GATE_USER, GATE_PASSWORD=GATE_PASSWORD,
                                      GATE_PORT=str(GATE_PORT)), start_new_session=True)
AUTH = (GATE_USER, GATE_PASSWORD)
for _ in range(40):
    if gate_proc.poll() is not None:
        raise RuntimeError("gate exited:\n" + tail(GATE_LOG))
    try:
        if requests.get(GATE + "/api/version", auth=AUTH, timeout=3).status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(0.5)
else:
    raise RuntimeError("gate did not answer within 20 s:\n" + tail(GATE_LOG))


def gate_checks(base):
    """The gate must refuse what it should refuse, and pass what the app needs. Shows each real status code."""
    def status(method, path, **kw):
        return requests.request(method, base + path, timeout=15, **kw).status_code
    checks = [
        ("no password -> 401", status("GET", "/api/tags"), 401),
        ("wrong password -> 401", status("GET", "/api/tags", auth=(GATE_USER, "wrong")), 401),
        ("model download blocked -> 403", status("POST", "/api/pull", auth=AUTH, json={"model": "x"}), 403),
        ("model delete blocked -> 403", status("DELETE", "/api/delete", auth=AUTH, json={"model": MODELS[0]}), 403),
        ("password -> 200", status("GET", "/api/tags", auth=AUTH), 200),
    ]
    results = {f"{name} (got {got})": got == want for name, got, want in checks}
    left = [x["name"] for x in requests.get(base + "/api/tags", auth=AUTH, timeout=15).json().get("models", [])]
    results["both models still there after the delete attempt"] = all(m in left for m in MODELS)
    return results


results = gate_checks(GATE)
for k, ok in results.items():
    print("ok  " if ok else "FAIL", k)
if not all(results.values()):
    raise RuntimeError("gate checks failed")

In [ ]:
# 7) Public HTTPS link to the gate (Cloudflare quick tunnel). Every run replaces the link.
TUNNEL_LOG = "/content/cloudflared.log"


def tunnel_url():
    found = re.findall(r"https://[a-z0-9-]+[.]trycloudflare[.]com", tail(TUNNEL_LOG, 200000))
    found = [u for u in found if not u.startswith("https://api.")]
    return found[-1] if found else None


if not shutil.which("cloudflared"):
    sh("curl -fsSL -o /usr/local/bin/cloudflared "
       "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
       " && chmod +x /usr/local/bin/cloudflared")
print(sh("cloudflared --version").strip())

PUBLIC_URL = None
subprocess.run(["pkill", "-f", "cloudflared tunnel"])  # close any older link first
time.sleep(1)
open(TUNNEL_LOG, "w").close()
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", GATE],
                               stdout=open(TUNNEL_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(90):
    if tunnel_url():
        break
    if tunnel_proc.poll() is not None:
        raise RuntimeError("cloudflared exited:\n" + tail(TUNNEL_LOG))
    time.sleep(1)
else:
    tunnel_proc.terminate()
    raise RuntimeError("no tunnel link after 90 s (tunnel stopped). Run this cell again:\n" + tail(TUNNEL_LOG))
PUBLIC_URL = tunnel_url()
print("tunnel:", PUBLIC_URL)

In [ ]:
# 8) Check from the outside, through the public link, exactly like the app will
for attempt in range(1, 31):              # a brand-new link can take 10-60 s before it resolves
    try:
        code = requests.get(PUBLIC_URL + "/api/version", auth=AUTH, timeout=10).status_code
        if code == 200:
            break
        print(f"attempt {attempt}: HTTP {code}")
    except requests.RequestException as e:
        print(f"attempt {attempt}: {type(e).__name__}")
    time.sleep(3)
else:
    raise RuntimeError("the public link never answered 200. Run cell 7 again, then this cell.")

results = gate_checks(PUBLIC_URL)
for k, ok in results.items():
    print("ok  " if ok else "FAIL", k, "(through the link)")
if not all(results.values()):
    raise RuntimeError("gate checks through the link failed")

host = PUBLIC_URL.split("://", 1)[1]
BASE_URL_WITH_PASSWORD = f"https://{GATE_USER}:{GATE_PASSWORD}@{host}"
for m in MODELS:   # credentials inside the URL, the way the app will send them
    secs, reply = app_chat(BASE_URL_WITH_PASSWORD, m, timeout=90)
    print(f"through the link: {m} {secs:.1f}s, reply {reply[:60]}")

print()
print("=" * 78)
print("Copy the next line into D:\\TenderMind\\.env  (open it with: notepad D:\\TenderMind\\.env)")
print("Do NOT paste it into the chat: it contains the password.")
print()
print("OLLAMA_BASE_URL=" + BASE_URL_WITH_PASSWORD)
print("=" * 78)

In [ ]:
# 9) Optional: status every minute. Set WATCH = True and run this cell; stopping it does NOT close the link.
WATCH = False
while WATCH:
    try:
        link = requests.get(PUBLIC_URL + "/api/version", auth=AUTH, timeout=10).status_code
    except requests.RequestException as e:
        link = type(e).__name__
    tunnel = "running" if tunnel_proc.poll() is None else f"EXITED({tunnel_proc.returncode})"
    gate = "running" if gate_proc.poll() is None else f"EXITED({gate_proc.returncode})"
    print(time.strftime("%H:%M:%S"), f"link={link} ollama={ollama_version() or 'DOWN'} gate={gate} tunnel={tunnel}")
    time.sleep(60)
print("watching is off (set WATCH = True to turn it on)")

In [ ]:
# 10) Close the public link when you are done: set CLOSE = True and run this cell
CLOSE = False
if CLOSE:
    subprocess.run(["pkill", "-f", "cloudflared tunnel"])
    subprocess.run(["pkill", "-f", GATE_SCRIPT])
    time.sleep(1)
    left = subprocess.run(["pgrep", "-af", "cloudflared tunnel|ollama_gate"], capture_output=True, text=True).stdout
    print("link closed" if not left.strip() else "still running:\n" + left)
else:
    print("link is still open (set CLOSE = True and run this cell to close it)")